# Day 03 - Polygons | Olive Landscapes of Córdoba

Exploring Córdoba’s connection to olive growing, a landscape linked to my family, through a choropleth map showing olive groves as a percentage of each municipality’s total area.

This notebook adapts Ben Dexter Cooley’s GeoPandas and Matplotlib tutorial to explore differences in olive-growing coverage across the province.

**Inspiration:** [Let’s make a map! Using GeoPandas and Matplotlib](https://github.com/bendoesdata/make-a-map-geopandas)

**Tools:** Python, GeoPandas, Pandas and Matplotlib.

**#30DayMapChallenge 2026 · Day 03 · Polygons**

In [ ]:
%pip install pandas geopandas matplotlib requests

In [ ]:
%pip install xlrd

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile

import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.ticker import PercentFormatter

In [ ]:
# Set project folders relative to the notebook's working directory
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

# Create folders if they do not already exist
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data folder: {DATA_DIR}")
print(f"Output folder: {OUTPUT_DIR}")

In [ ]:
# Download Córdoba municipal boundaries from the IGN API
BOUNDARIES_URL = (
    "https://api-features.ign.es/"
    "collections/administrativeunit/items"
)
BOUNDARIES_PATH = DATA_DIR / "cordoba_municipalities_ign.geojson"

if not BOUNDARIES_PATH.exists():
    params = {
        "f": "json",
        "limit": 100,
        "nationallevelname": "Municipio",
        "codnut3": "ES613",
    }

    print("Downloading Córdoba municipal boundaries from IGN...")

    response = requests.get(
        BOUNDARIES_URL,
        params=params,
        timeout=180,
    )
    response.raise_for_status()
    data = response.json()

    # Check that the response contains all matching municipalities
    if not data.get("features"):
        raise ValueError("The IGN response contains no municipalities.")

    if any(link.get("rel") == "next" for link in data.get("links", [])):
        raise ValueError("The response is paginated; more features are available.")

    municipalities = gpd.GeoDataFrame.from_features(
        data["features"],
        crs="EPSG:4326",
    )

    municipalities.to_file(BOUNDARIES_PATH, driver="GeoJSON")
else:
    print("Using previously downloaded IGN boundaries.")
    municipalities = gpd.read_file(BOUNDARIES_PATH)

# Extract the five-digit municipality code for joining agricultural data
municipalities["municipality_code"] = (
    municipalities["nationalcode"].astype(str).str[-5:]
)

print(f"Municipalities loaded: {len(municipalities)}")
display(municipalities[["municipality_code", "nameunit"]].head())

In [ ]:
# Download official municipal crop statistics for 2023
CROPS_URL = (
    "https://www.juntadeandalucia.es/sites/default/"
    "files/2025-04/WEB%201TM%202023.zip"
)
CROPS_PATH = DATA_DIR / "municipal_crops_2023.zip"

if not CROPS_PATH.exists():
    print("Downloading municipal crop statistics...")

    response = requests.get(CROPS_URL, timeout=180)
    response.raise_for_status()

    # Validate the ZIP archive before saving it
    with ZipFile(BytesIO(response.content)) as archive:
        crop_files = archive.namelist()

    CROPS_PATH.write_bytes(response.content)
    print("Download complete.")
else:
    print("Using previously downloaded crop statistics.")

# List the provincial files included in the archive
with ZipFile(CROPS_PATH) as archive:
    for filename in archive.namelist():
        print(filename)

In [ ]:
# Locate the Córdoba workbook despite the ZIP filename encoding
with ZipFile(CROPS_PATH) as archive:
    matching_files = [
        name for name in archive.namelist()
        if name.lower().endswith(".xls") and "RDOBA" in name.upper()
    ]

    if len(matching_files) != 1:
        raise ValueError("Could not uniquely identify the Córdoba workbook.")

    cordoba_excel = BytesIO(archive.read(matching_files[0]))

# Inspect workbook sheets before loading the crop table
workbook = pd.ExcelFile(cordoba_excel, engine="xlrd")

print("Available sheets:", workbook.sheet_names)

preview = pd.read_excel(workbook, sheet_name=0, header=None)
display(preview.head(15))

In [ ]:
# Load the crop table using the first row as column headers
crops = pd.read_excel(workbook, sheet_name=0, header=0)

# Remove empty columns and standardize column names
crops = crops.dropna(axis=1, how="all")
crops.columns = crops.columns.astype(str).str.strip()

# Inspect all olive-related crop categories before selecting them
olive_mask = crops["CULTIVO"].astype(str).str.contains(
    "oliv|aceitun",
    case=False,
    na=False,
)

olive_categories = (
    crops.loc[
        olive_mask,
        ["GRUPO_CULTIVO", "CULTIVO", "CLAVE_CULTIVO"],
    ]
    .drop_duplicates()
    .sort_values("CLAVE_CULTIVO")
)

display(olive_categories)

In [ ]:
# Select the overall olive-growing category
crop_codes = pd.to_numeric(crops["CLAVE_CULTIVO"], errors="coerce")
olive_data = crops.loc[
    crop_codes.eq(1400),
    ["MUN_INE", "MUNICIPIO", "TOTAL_HA"],
].copy()

# Standardize municipality codes and olive-growing areas
olive_data["municipality_code"] = (
    pd.to_numeric(olive_data["MUN_INE"], errors="raise")
    .astype("int64")
    .astype(str)
    .str.zfill(5)
)

olive_data["olive_area_ha"] = pd.to_numeric(
    olive_data["TOTAL_HA"], errors="raise"
)

# Join agricultural statistics to municipal boundaries
olive_map = municipalities.merge(
    olive_data[["municipality_code", "olive_area_ha"]],
    on="municipality_code",
    how="left",
    validate="one_to_one",
)

# Stop if any municipality lacks agricultural data
missing = olive_map.loc[
    olive_map["olive_area_ha"].isna(), "nameunit"
]
if not missing.empty:
    raise ValueError(f"Missing olive-growing data: {missing.tolist()}")

# Calculate municipal areas in an equal-area projection
olive_map = olive_map.to_crs("EPSG:3035")
olive_map["municipal_area_ha"] = olive_map.geometry.area / 10_000

# Express olive-growing area as a percentage of total municipal area
olive_map["olive_pct"] = (
    olive_map["olive_area_ha"] / olive_map["municipal_area_ha"] * 100
)

print(f"Municipalities matched: {len(olive_map)}")
print(
    f"Olive coverage range: "
    f"{olive_map['olive_pct'].min():.1f}%–"
    f"{olive_map['olive_pct'].max():.1f}%"
)

display(
    olive_map[
        ["nameunit", "olive_area_ha", "municipal_area_ha", "olive_pct"]
    ]
    .sort_values("olive_pct", ascending=False)
    .round(2)
    .head(10)
)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from matplotlib.font_manager import FontProperties

# Project the map to UTM zone 30N for a scale bar in metres
plot_map = olive_map.to_crs("EPSG:25830")

# Define a custom scale from light green to dark green
olive_cmap = LinearSegmentedColormap.from_list(
    "olive_greens",
    ["#b7cd7f", "#203500"],
    N=256,
)

# Create the choropleth figure
fig, ax = plt.subplots(figsize=(10, 10), facecolor="#f0edf8")
ax.set_facecolor("#f0edf8")

# Map olive-growing area using the custom green scale
plot_map.plot(
    column="olive_pct",
    cmap=olive_cmap,
    vmin=0,
    vmax=100,
    linewidth=0.4,
    edgecolor="#ffffff",
    ax=ax,
)

ax.set_axis_off()

# Apply the shared challenge title format
fig.suptitle(
    "Day 3 - Polygons\nOlive Landscapes of Córdoba",
    fontsize=24,
    color="#203500",
    y=0.95,
)

# Use the same color scale for the map and its legend
sm = plt.cm.ScalarMappable(
    norm=Normalize(vmin=0, vmax=100),
    cmap=olive_cmap,
)

cbar = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.03)
cbar.set_label("Municipal area covered by olive groves (%)")
cbar.ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))

# Add the shared challenge footer
fig.text(
    0.5, 0.04,
    "#30DayMapChallenge | 2026 | Day - 03 | "
    "Olive Landscapes of Córdoba | Map by Maria Victoria León Parra",
    ha="center",
    fontsize=9,
    color="#203500",
)

# Credit the data sources
fig.text(
    0.5, 0.06,
    "Crop statistics: Junta de Andalucía, 2023 | "
    "Municipal boundaries: © IGN",
    ha="center",
    fontsize=8,
    color="#203500",
)

# Reserve space for the title and footer
fig.subplots_adjust(top=0.88, bottom=0.08)

# Add an arrow indicating grid north
ax.annotate(
    "N",
    xy=(0.90, 0.94),
    xytext=(0.90, 0.82),
    xycoords="axes fraction",
    textcoords="axes fraction",
    ha="center",
    va="center",
    fontsize=14,
    fontweight="bold",
    color="#203500",
    arrowprops={
        "arrowstyle": "-|>",
        "facecolor": "#203500",
        "edgecolor": "#203500",
        "linewidth": 1.5,
    },
)

# Add a 20 km scale bar using projected map coordinates
scale_bar = AnchoredSizeBar(
    ax.transData,
    20_000,
    "20 km",
    loc="lower left",
    pad=0.6,
    borderpad=1,
    sep=6,
    frameon=False,
    size_vertical=500,
    color="#203500",
    fontproperties=FontProperties(size=10),
)
ax.add_artist(scale_bar)

plt.show()

In [ ]:
# Export the map while preserving its background and layout
map_path = OUTPUT_DIR / "day03_polygons_olive_cordoba.png"

fig.savefig(
    map_path,
    dpi=300,
    facecolor=fig.get_facecolor(),
    bbox_inches="tight",
    pad_inches=0.2,
)

print(f"Map saved to: {map_path}")

## Sources and limitations

### Sources

- **Municipal boundaries:** Instituto Geográfico Nacional (IGN), downloaded through its [API Features service](https://api-features.ign.es/collections/administrativeunit).
- **Agricultural statistics:** Junta de Andalucía, [Municipal crop areas in Andalucía, 2023](https://www.juntadeandalucia.es/servicios/estadistica-cartografia/actividad/detalle/71175/473605.html). Córdoba workbook, GENERAL sheet, Olivar category (1400).
- **Tutorial:** [Let’s make a map! Using GeoPandas and Matplotlib](https://github.com/bendoesdata/make-a-map-geopandas) by Ben Dexter Cooley.

### Limitations

Olive-growing area is expressed as a percentage of each municipality’s total area, including non-agricultural land. Municipal colors represent an aggregate percentage, not the precise locations of olive groves.

Agricultural statistics refer to 2023. IGN boundaries were downloaded when the notebook was run and may reflect later boundary updates.

Municipal areas were calculated in EPSG:3035. The final map uses ETRS89 / UTM zone 30N (EPSG:25830), with a 20 km scale bar and a grid north arrow.

All 77 municipalities were matched to agricultural records. Calculated olive coverage ranges from approximately 0.1% to 93.6%.

### Reproducibility

The notebook downloads the source datasets automatically and reuses local copies when available. Downloaded files in `data/` should be excluded from Git; the notebook retains the source URLs and processing steps.